# Machine Learning (ML) dataset generation

This notebook walks through generating machine learning data for each region and season of interest at a long lead time (4-8 months)

Here, 3 datasets are generated for each region and season.

- Monthly
    - For all years, we subset the months from the seasons of interest, assign a tercile label based on CHIRPS observations, and compute SST predictors at a lead time of 6-8 months based on that season.
    - For example, if we are interested in MAM, months 3, 4, and 5 are assigned a tercile label (AN/BN/N) based on CHIRPS, and we will compute all the SST anomaly predictors (i.e Nino4, IOD_west, etc.) that are 6, 7, and 8 months before the start of MAM, and assign them to months 3, 4, and 5 respectively.
- Seasonal
    - For all years, we compute the seasonal average, assign a tercile label based on CHIRPS, and compute SST predictors at a lead time of 4-6 months based on that season and year.
    - For example, if we are interested in MAM, we take the average observed precip of MAM, assign a tercile label, and compute all the SST anomaly predictors that are 4-8 months before the start of MAM, and assign them to the season.
- Monthly, with seasonal Tercile categories
    - Same conditions as the Monthly ML data, except we assign the tercile label of the season to each month.
    - For example, if we are interested in MAM, and the months 3, 4, and 5 have the observed terciles BN, AN, and AN, but the seasonal average for MAM was AN, we assign 3, 4, and 5 the AN tercile. This seems counterintuitive, but it improves model predictions (WIP on why that happens).


In [1]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import os

In [2]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# set to your file paths
sst_file_path = '/content/drive/My Drive/capstone_data/SST/sst.mnmean.nc'
chirps_file_path = '/content/drive/My Drive/capstone_data/CHIRPS/chirps-v2.0.monthly.nc'
ml_csv_folder_save_path = '/content/drive/My Drive/capstone_data/ml_data_automation_test'

In [4]:
# open sst data, subset to 1993 to 2024, extract month and year
sst = xr.open_dataset(sst_file_path)
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.sel(time=slice('1992-01-01', '2024-12-31')).to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year

In [5]:
# open CHIRPS
chirps = xr.open_dataset(chirps_file_path)

In [6]:
# compute standard deviation of sst data
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [7]:
# Compute sst anomalies
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [8]:
# dictionary for region and season info
region_seasons_months_dict = {
    'eastern_east_africa': {'seasons': {'MAM': [3, 4, 5],
                                        'OND': [10, 11, 12]},
                            'latitude': (-3.5, 8),
                            'longitude':(38, 50)},
    'lake_victoria_basin': {'seasons': {'SON': [9, 10, 11],
                                        'MAM': [3, 4, 5],
                                        'DJF': [12, 1, 2]},
                            'latitude': (-4, 1.55),
                            'longitude':(29, 36)},
    'west_africa': {'seasons': {'JAS': [7, 8, 9]},
                            'latitude': (10, 13.5),
                            'longitude':(-10, 0)},
    'south_sudan': {'seasons': {'MJJ': [5, 6, 7],
                                        'JAS': [7, 8, 9],
                                        'ASO': [8, 9, 10]},
                            'latitude': (3.5, 12.5),
                            'longitude':(25, 35)},
    'eastern_ukraine': {'seasons': {'JA': [7, 8],
                                        'AMJ': [4, 5, 6],
                                        'DJF': [12, 1, 2]},
                            'latitude': (45, 51),
                            'longitude':(31, 40)},
    'sri_lanka': {'seasons': {'OND': [10, 11, 12]},
                            'latitude': (5.5, 10),
                            'longitude':(79, 82)},
    'southern_africa': {'seasons': {'FMA': [2, 3, 4],
                                    'DJF': [12, 1, 2]},
                            'latitude': (-23, -15),
                            'longitude':(25, 34)}
}

In [9]:
'''Subset CHIRPS to region of interest, map seasonal months accordingly, and
compute monthly and seasonal means
'''
# dictionary to store monthly and seasonal mapped dataframes regionally
chirps_mapped_dfs = {}

# iterate over region name and region info from region_seasons_months_dict
for region_name, region_info in region_seasons_months_dict.items():
        # unpack lat and lon
        lat_min, lat_max = region_info['latitude']
        lon_min, lon_max = region_info['longitude']

        # slice chirps and create relevant columns
        chirps_current_region = chirps.sel(time=slice('1993-01-01', '2024-12-01'), latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max)).to_dataframe().reset_index()
        chirps_current_region['month'] = chirps_current_region['time'].dt.month
        chirps_current_region['year'] = chirps_current_region['time'].dt.year

        # for each season of that region

        # Build month-to-season mapping
        season_month_map = {
            month: season
            for season, months in region_info['seasons'].items()
            for month in months
        }

        # map chirps using month-to-season mapping
        chirps_current_region['season'] = chirps_current_region['month'].map(season_month_map)

        chirps_current_region = chirps_current_region.dropna(subset=['season'])

        chirps_current_region.dropna(inplace=True)

        chirps_current_region['region'] = region_name

        # compute monthly means
        chirps_current_region_monthly = chirps_current_region.groupby(['year', 'month', 'region'])['precip'].mean().reset_index()

        # compute seasonal means
        chirps_current_region_season = chirps_current_region.groupby(['year', 'season', 'region'])[['precip']].mean().reset_index()

        # add to mapped dfs dict
        chirps_mapped_dfs[region_name] = {'monthly': chirps_current_region_monthly, 'seasonal': chirps_current_region_season}


### Now, we have seasonal and monthly CHIRPS data for each region, subsetted to seasons of interest. We move on to assigning tercile categories for each month and season.

In [10]:
def get_tercile_labels_by_season(chirps):
    # Initialize an empty column for tercile
    chirps['tercile'] = None

    # Group by season and assign terciles within each group
    for season, group in chirps.groupby('season'):
        # Calculate the tercile thresholds for this season
        terciles = group['precip'].quantile([0.33, 0.66])
        low, high = terciles[0.33], terciles[0.66]

        # Assign terciles based on thresholds
        def assign_tercile(p):
            if p <= low:
                return 'bn'  # Below Normal
            elif p <= high:
                return 'n'   # Normal
            else:
                return 'an'  # Above Normal

        # Apply to rows of the original DataFrame
        chirps.loc[group.index, 'tercile'] = group['precip'].apply(assign_tercile)

    return chirps


In [11]:
'''For each region, compute tercile categories for each month/year combo and
label accordingly. Then, store into a dictionary.

monthly_tercile_dict structure:
{region_name: {(month,): dataframe}}

where the dataframe columns are: [year, month, region, precip, tercile]

Example: In 1993, EEA experienced AN/BN/N rains during March
when compared to overall rainfall in 1993.
'''
# initiate dictionary to store month/year tercile combination data regionally
monthly_tercile_dict = {}

# iterate over region name and df dict in mapped dataframes dict
for region_name, df_dict in chirps_mapped_dfs.items():
    # extract the monthly mapped chirps data for the current region
    chirps_current_region_monthly = df_dict['monthly']

    # dictionary to hold month-wise tercile dfs for this region
    region_monthly_terciles = {}

    # iterate over unique combinations of months
    for month, df in chirps_current_region_monthly.groupby(['month']):
        # Compute terciles
        terciles = df['precip'].quantile([0.33, 0.66]).to_list()
        bn_years = df.query(f'precip <= {terciles[0]}')['year'].to_list()
        n_years = df.query(f'{terciles[0]} < precip <= {terciles[1]}')['year'].to_list()
        an_years = df.query(f'precip > {terciles[1]}')['year'].to_list()

        # Map years to tercile category
        year_to_category = {
            year: 'bn' for year in bn_years
        } | {
            year: 'n' for year in n_years
        } | {
            year: 'an' for year in an_years
        }

        # Assign tercile labels
        df['tercile'] = df['year'].map(year_to_category)

        # Store df by month
        region_monthly_terciles[month] = df

    # Store month-wise dfs under region key
    monthly_tercile_dict[region_name] = region_monthly_terciles

In [12]:
'''For every region in monthly_tercile_dict, obtain the full combined labeled
monthly data and store into a dictionary.

labeled_chirps_monthly structure:
{region_name: dataframe}

where the dataframe columns are: [year, month, region, precip, tercile]
'''
labeled_chirps_monthly_dfs = {}

# Iterate over every region name and dictionary of separate month dataframes in monthly_tercile_dict
for region_name, monthly_tercile_df_dict in monthly_tercile_dict.items():

    # combine separate month dataframes into one dataframe
    labeled_monthly_df = pd.concat(monthly_tercile_df_dict.values()).reset_index().drop('index', axis = 1)

    # store combined dataframe into labeled_chirps_monthly_dfs with key as region_name
    labeled_chirps_monthly_dfs[region_name] = labeled_monthly_df

In [13]:
'''For every region, obtain the full combined labeled
seasonal data and store into a dictionary.

labeled_chirps_seasonal structure:
{region_name: dataframe}

where the dataframe columns are: [year, season, region, precip, tercile]
'''
labeled_chirps_seasonal_dfs = {}

# iterate over every region name and df_dict (monthly and seasonal data in here) in chirps_mapped_dfs
for region_name, df_dict in chirps_mapped_dfs.items():
    # assign tercile labels for current seasonal dataframe, with key as region name
    labeled_chirps_seasonal_dfs[region_name] = get_tercile_labels_by_season(df_dict['seasonal'])

In [29]:
def merge_monthly_with_seasonal_terciles(monthly_df, seasonal_df, region, region_seasons_months_dict):
    """
    Assign seasonal tercile labels to monthly data using region-specific season definitions.

    Parameters:
        monthly_df (pd.DataFrame): DataFrame with at least ['year', 'month'] columns
        seasonal_df (pd.DataFrame): DataFrame with ['year', 'season', 'tercile']
        region (str): Region key from region_seasons_months_dict
        region_seasons_months_dict (dict): Mapping of region to season-months

    Returns:
        pd.DataFrame: monthly_df with added 'season' and 'tercile' columns
    """

    # Get region's season-month mapping
    season_month_map = region_seasons_months_dict[region]['seasons']

    # Create month-to-season lookup
    month_to_season = {}
    for season, months in season_month_map.items():
        for m in months:
            month_to_season[m] = season

    # Assign season column
    monthly_df = monthly_df.copy()
    monthly_df['season'] = monthly_df['month'].map(month_to_season)

    # Drop rows where month isn't part of any defined season
    monthly_df = monthly_df.dropna(subset=['season'])

    # Merge with seasonal tercile data on year and season
    merged_df = monthly_df.merge(seasonal_df[['year', 'season', 'tercile']],
                                 how='left', on=['year', 'season'])

    return merged_df


In [34]:
'''For every region, obtain the full combined labeled
monthly data with seasonal tercile assignment, and store into a dictionary.

labeled_chirps_seasonal structure:
{region_name: dataframe}

where the dataframe columns are: [year, season, region, precip, tercile]
'''

# Dictionary to store labeled monthly data with seasonal terciles
labeled_chirps_monthly_seasonal_tercile_dfs = {}

# Loop over each region
for region_name, df_dict in chirps_mapped_dfs.items():
    # Get the monthly and seasonal labeled data
    monthly_df = df_dict['monthly']
    seasonal_labeled_df = labeled_chirps_seasonal_dfs[region_name]

    # Merge and label
    labeled_monthly = merge_monthly_with_seasonal_terciles(
        monthly_df=monthly_df,
        seasonal_df=seasonal_labeled_df,
        region=region_name,
        region_seasons_months_dict=region_seasons_months_dict
    )

    # Store in dictionary
    labeled_chirps_monthly_seasonal_tercile_dfs[region_name] = labeled_monthly


In [14]:
# extract labeld EEA monthly tercile data
labeled_chirps_monthly_eea = labeled_chirps_monthly_dfs['eastern_east_africa']

In [32]:
# extract labeled EEA seasonal tercile data
labeled_chirps_seasonal_eea = labeled_chirps_seasonal_dfs['eastern_east_africa']

In [33]:
# extract labeled EEA monthly data, with seasonal terciles (works better in xgboost compared to monthly, idk why)
labeled_chirps_monthly_seasonal_tercile_eea = merge_monthly_with_seasonal_terciles(chirps_mapped_dfs['eastern_east_africa']['monthly'], labeled_chirps_seasonal_eea, 'eastern_east_africa', region_seasons_months_dict)

In [36]:
labeled_chirps_monthly_eea

,year,month,region,precip,tercile
0,1993,3,eastern_east_africa,12.088976,bn
1,1994,3,eastern_east_africa,16.170179,bn
2,1995,3,eastern_east_africa,39.480980,an
3,1996,3,eastern_east_africa,36.686726,an
4,1997,3,eastern_east_africa,36.074913,an
...,...,...,...,...,...
187,2020,12,eastern_east_africa,15.553253,bn
188,2021,12,eastern_east_africa,41.772377,an
189,2022,12,eastern_east_africa,15.340866,bn
190,2023,12,eastern_east_africa,21.531441,n


In [37]:
labeled_chirps_seasonal_eea

,year,season,region,precip,tercile
0,1993,MAM,eastern_east_africa,63.102024,n
1,1993,OND,eastern_east_africa,39.483353,bn
2,1994,MAM,eastern_east_africa,70.791168,n
3,1994,OND,eastern_east_africa,72.053864,an
4,1995,MAM,eastern_east_africa,75.360275,an
...,...,...,...,...,...
59,2022,OND,eastern_east_africa,41.309280,bn
60,2023,MAM,eastern_east_africa,92.914536,an
61,2023,OND,eastern_east_africa,131.538925,an
62,2024,MAM,eastern_east_africa,88.177818,an


In [38]:
labeled_chirps_monthly_seasonal_tercile_eea

,year,month,region,precip,season,tercile
0,1993,3,eastern_east_africa,12.088976,MAM,n
1,1993,4,eastern_east_africa,69.710571,MAM,n
2,1993,5,eastern_east_africa,107.506531,MAM,n
3,1993,10,eastern_east_africa,62.086601,OND,bn
4,1993,11,eastern_east_africa,35.567661,OND,bn
...,...,...,...,...,...,...
187,2024,4,eastern_east_africa,167.127075,MAM,an
188,2024,5,eastern_east_africa,73.994225,MAM,an
189,2024,10,eastern_east_africa,47.677654,OND,n
190,2024,11,eastern_east_africa,82.150604,OND,n


In [39]:
'''Extract SST Anomalies from SST data
'''

nino_34 = sst_anomaly.query('-5 <= lat <= 5 and -170<= lon <= -120').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().rename({'normalized_sst_anomaly': 'nino_34'}, axis=1)

nino_4 = sst_anomaly.query('-5 <= lat <= 5 and lon <= -150 or -5 <= lat <= 5 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'nino_4'}, axis=1)

western_west_v = sst_anomaly.query('-15 <= lat <= 20 and 120 <= lon <= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'western_west_v'}, axis=1)

northern_west_v = sst_anomaly.query('20 <= lat <= 35 and lon <= -150 or 20 <= lat <= 35 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'northern_west_v'}, axis=1)

southern_west_v = sst_anomaly.query('-30 <= lat <= -15 and lon <= -150 or -30 <= lat <= -15 and lon >= 155').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'southern_west_v'}, axis=1)

SWIO = sst_anomaly.query('-50 <= lat <= -20 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'SWIO'}, axis=1)

IOD_west = sst_anomaly.query('-10 <= lat <= 10 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_west'}, axis=1)

IOD_east = sst_anomaly.query('-10 <= lat <= 0 and 90 <= lon <= 110').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_east'}, axis=1)

predictors = pd.concat([nino_34, nino_4, western_west_v, northern_west_v, southern_west_v, SWIO, IOD_west, IOD_east], axis=1)

In [40]:
def create_lead_month_mapping_seasonal(target_season_name, target_start_month, lead_time):
    """
    Calculates the prediction month for a given target season and lead time.

    Args:
        target_season_name (str): The name of the target season (e.g., 'MAM').
        target_start_month (int): The numerical start month (1-12) of the target season.
        lead_time (int): The number of months lead time (e.g., 4).

    Returns:
        dict: A dictionary mapping the prediction month (int) to the target season name (str).
              Example: {11: 'MAM'} for a 4-month lead to March.
    """
    if not 1 <= target_start_month <= 12:
        raise ValueError("target_start_month must be between 1 and 12")
    if lead_time < 0:
        raise ValueError("lead_time cannot be negative")

    # Calculate the prediction month (1-12)
    # (target_start_month - lead_time - 1) gives the zero-based index offset
    # % 12 handles the wrap-around for negative results
    # + 1 converts back to 1-based month index
    prediction_month = (target_start_month - lead_time - 1) % 12 + 1

    return {prediction_month: target_season_name}

In [41]:
'''Generates lead month mappings for all regions and seasons and stores it
into all_region_season_lead_maps.

all_region_season_lead_maps structure:
{region_name: {season_name: {lead_time: {lead_month_mapping: season_name}}}}
'''

region_season_start_info = {'eastern_east_africa': {'MAM': 3, 'OND': 10},
                            'lake_victoria_basin': {'DJF': 12, 'SON': 9, 'MAM': 3},
                            'west_africa': {'JAS': 7},
                            'south_sudan': {'MJJ':5, 'JAS':7, 'ASO':8},
                            'eastern_ukraine': {'JA': 7, 'AMJ': 4, 'DJF': 12},
                            'sri_lanka': {'OND': 10},
                            'southern_africa': {'FMA': 2, 'DJF': 12}}

# All mappings organized by region and season
all_region_season_lead_maps = {}

# iterate over every region and start month info
for region, seasons_start_dict in region_season_start_info.items():
    # assign the region key
    all_region_season_lead_maps[region] = {}

    # for every season and start month
    for season, start_month in seasons_start_dict.items():

        # create a lead map dictionary
        current_region_season_lead_maps = {}

        # create lead maps for the season, region, and lead
        for lead in range(4, 9): # Leads from 4 to 8 inclusive
            current_region_season_lead_maps[lead] = create_lead_month_mapping_seasonal(season, start_month, lead)

        # assign lead maps to region and season key
        all_region_season_lead_maps[region][season] = current_region_season_lead_maps

all_region_season_lead_maps

{'eastern_east_africa': {'MAM': {4: {11: 'MAM'},
   5: {10: 'MAM'},
   6: {9: 'MAM'},
   7: {8: 'MAM'},
   8: {7: 'MAM'}},
  'OND': {4: {6: 'OND'},
   5: {5: 'OND'},
   6: {4: 'OND'},
   7: {3: 'OND'},
   8: {2: 'OND'}}},
 'lake_victoria_basin': {'DJF': {4: {8: 'DJF'},
   5: {7: 'DJF'},
   6: {6: 'DJF'},
   7: {5: 'DJF'},
   8: {4: 'DJF'}},
  'SON': {4: {5: 'SON'},
   5: {4: 'SON'},
   6: {3: 'SON'},
   7: {2: 'SON'},
   8: {1: 'SON'}},
  'MAM': {4: {11: 'MAM'},
   5: {10: 'MAM'},
   6: {9: 'MAM'},
   7: {8: 'MAM'},
   8: {7: 'MAM'}}},
 'west_africa': {'JAS': {4: {3: 'JAS'},
   5: {2: 'JAS'},
   6: {1: 'JAS'},
   7: {12: 'JAS'},
   8: {11: 'JAS'}}},
 'south_sudan': {'MJJ': {4: {1: 'MJJ'},
   5: {12: 'MJJ'},
   6: {11: 'MJJ'},
   7: {10: 'MJJ'},
   8: {9: 'MJJ'}},
  'JAS': {4: {3: 'JAS'},
   5: {2: 'JAS'},
   6: {1: 'JAS'},
   7: {12: 'JAS'},
   8: {11: 'JAS'}},
  'ASO': {4: {4: 'ASO'},
   5: {3: 'ASO'},
   6: {2: 'ASO'},
   7: {1: 'ASO'},
   8: {12: 'ASO'}}},
 'eastern_ukraine': {'JA':

## Now each region has its respective season and lead mapping for each season for lead of 4 to 8 months

We move on to using these maps to subset our predictors by lead time. For example, in EEA MAM 2023 season, all the lead 4 predictors (SST predictors in November 2022) will have a dataframe

In [42]:
'''Subset SST predictors by lead time into a processed_dfs dictionary

processed_dfs structure: {region_name: {season_name: {lead_time: dataframe}}}
'''

processed_dfs = {}

for region in all_region_season_lead_maps.keys():
    # initiate dictionary for region
    processed_dfs[region] = {}
    for season, lead_mapping in all_region_season_lead_maps[region].items():
        # initiate dictionary for region's season
        processed_dfs[region][season] = {}
        for lead_time in range(4, 9):
            print(f"Processing lead time: {lead_time} for {region} {season}")

            # Check if the mapping exists for the current lead time
            if lead_time not in lead_mapping:
                print(f"Warning: Mapping for lead time {lead_time} not found. Skipping.")
                continue

            current_map = lead_mapping[lead_time]

            # 1. Copy the base predictors DataFrame
            temp_df = predictors.copy()

            # 2. Map 'effect_season' using the correct lead time map
            temp_df['effect_season'] = temp_df['month'].map(current_map)

            season_start = region_season_start_info[region][season]

            year_adjustment = 0
            if lead_time >= season_start: # Check if lead time crosses start of year boundary
                year_adjustment = 1

            if 'year' in temp_df.columns:
                temp_df['effect_year'] = temp_df['year'] + year_adjustment
            else:
                print(f"Warning: 'year' column not found for 'effect_year' calculation for lead {lead_time}.")

            # 3. Drop rows where mapping failed (NaN in 'effect_season') and drop 'month' column
            temp_df = temp_df.dropna(subset=['effect_season'])
            # Only drop 'month' if it exists, prevent errors
            if 'month' in temp_df.columns:
                temp_df = temp_df.drop('month', axis=1)
            else:
                print(f"Warning: 'month' column not found in temp_df for lead {lead_time} before dropping.")

            processed_dfs[region][season][f'lead_time_{lead_time}'] = temp_df

Processing lead time: 4 for eastern_east_africa MAM
Processing lead time: 5 for eastern_east_africa MAM
Processing lead time: 6 for eastern_east_africa MAM
Processing lead time: 7 for eastern_east_africa MAM
Processing lead time: 8 for eastern_east_africa MAM
Processing lead time: 4 for eastern_east_africa OND
Processing lead time: 5 for eastern_east_africa OND
Processing lead time: 6 for eastern_east_africa OND
Processing lead time: 7 for eastern_east_africa OND
Processing lead time: 8 for eastern_east_africa OND
Processing lead time: 4 for lake_victoria_basin DJF
Processing lead time: 5 for lake_victoria_basin DJF
Processing lead time: 6 for lake_victoria_basin DJF
Processing lead time: 7 for lake_victoria_basin DJF
Processing lead time: 8 for lake_victoria_basin DJF
Processing lead time: 4 for lake_victoria_basin SON
Processing lead time: 5 for lake_victoria_basin SON
Processing lead time: 6 for lake_victoria_basin SON
Processing lead time: 7 for lake_victoria_basin SON
Processing l

In [44]:
'''Rename and merge SST predictors for each region and season into
merged_predictors_by_region_season dictionary

merged_predictors_by_region_season structure: {region_name: {season_name: dataframe}}
'''
# initiate dictionary to store predictors by region, season and lead, unmerged
predictors_by_region_season = {}

# Initiate dictionary to store predictors by region and season, merged by lead
merged_predictors_by_region_season = {}

# Iterate over processed dfs
for region, season_lead_dict in processed_dfs.items():
    predictors_by_region_season[region] = {}
    merged_predictors_by_region_season[region] = {}
    for season, lead_dict in season_lead_dict.items():
        predictors_by_region_season[region][season] = []

        # Status
        print(f"Currently Processing {region} region, {season} season")

        # These columns MUST exist in all DataFrames inside processed_dfs.
        merge_keys = ['effect_season', 'effect_year']

        # List of lead times corresponding to the DataFrames in processed_dfs
        lead_times = list(range(4, 9)) # Corresponds to leads 4, 5, 6, 7, 8

        # Check if the number of dataframes matches the number of lead times
        if len(lead_dict.keys()) != len(lead_times):
            raise ValueError(f"Mismatch between number of dataframes ({len(processed_dfs)}) and lead times ({len(lead_times)})")

        # --- Step 2: Rename columns in each DataFrame (excluding merge keys) ---

        for key, df in lead_dict.items():
            lead = key.split('_')[-1]
            suffix = f"_L{lead}"

            # Create a copy to avoid modifying the original dfs in the list if needed later
            df_renamed = df.copy().drop(['year'], axis=1)

            # Check if all merge keys exist in the current DataFrame
            missing_keys = [key for key in merge_keys if key not in df_renamed.columns]
            if missing_keys:
                raise ValueError(f"Merge key(s) {missing_keys} not found in DataFrame for lead {lead}")

            # Rename columns that are NOT in merge_keys, do not rename region and season columns
            cols_to_rename = {col: f"{col}{suffix}" for col in df_renamed.columns if col not in merge_keys}
            df_renamed = df_renamed.rename(columns=cols_to_rename)

            predictors_by_region_season[region][season].append(df_renamed)

        # --- Step 3: Merge Horizontally ---

        # Start with the first DataFrame
        merged_predictors_seasonal = predictors_by_region_season[region][season][0]

        # Initiate empty dictionary for current region, current season
        merged_predictors_by_region_season[region][season] = {}

        # Iteratively merge the rest using an outer join
        for i in range(1, len(predictors_by_region_season[region][season])):
            try:
                merged_predictors_seasonal = pd.merge(
                    merged_predictors_seasonal,
                    predictors_by_region_season[region][season][i],
                    on=merge_keys,
                    how='outer' # Use 'outer' to keep all rows from all lead times
                                # Use 'inner' if you only want rows present in ALL lead times
                )

                # assign merged data to its respective region and season in the merged dict
                # updates after each merge.
                merged_predictors_by_region_season[region][season] = merged_predictors_seasonal

            except KeyError as e:
                print(f"\nError merging DataFrame for lead {lead_times[i]}. Missing key(s): {e}")
                print(f"Columns in left df: {merged_predictors_seasonal.columns.tolist()}")
                print(f"Columns in right df: {predictors_by_region_season[region][season][i].columns.tolist()}")
                # Handle error appropriately, e.g., break or continue

        print(f"Successfully Merged all predictors for {region} region {season} season.")


Currently Processing eastern_east_africa region, MAM season
Successfully Merged all predictors for eastern_east_africa region MAM season.
Currently Processing eastern_east_africa region, OND season
Successfully Merged all predictors for eastern_east_africa region OND season.
Currently Processing lake_victoria_basin region, DJF season
Successfully Merged all predictors for lake_victoria_basin region DJF season.
Currently Processing lake_victoria_basin region, SON season
Successfully Merged all predictors for lake_victoria_basin region SON season.
Currently Processing lake_victoria_basin region, MAM season
Successfully Merged all predictors for lake_victoria_basin region MAM season.
Currently Processing west_africa region, JAS season
Successfully Merged all predictors for west_africa region JAS season.
Currently Processing south_sudan region, MJJ season
Successfully Merged all predictors for south_sudan region MJJ season.
Currently Processing south_sudan region, JAS season
Successfully M

In [45]:
'''Merge the merged seasonal SST predictors with the labeled seasonal CHIRPS data,
save file

We merge one season at a time, and each region and season will have a file for ML data

Example: lake_victoria_basin_DJF_ml_data_seasonal.csv
Columns: year, season, region, precip, tercile, nino34_L4 ... IOD_east_L8, effect_season, effect_year
'''
# extract and save the ML data for each season and region

# iterate over region and season_dict in merged_predictors dict
for region, season_dict in merged_predictors_by_region_season.items():

    # for the current region, get the labeled CHIRPS data
    labeled_chirps_seasonal_current_region = labeled_chirps_seasonal_dfs[region]

    # Iterate over all the seasons in the current region, get all the merged predictors
    for season, merged_predictors_seasonal_data in season_dict.items():

        # merge the current seasonal predictors to the labeled chirps seasonal data
        ml_data_seasonal = labeled_chirps_seasonal_current_region.merge(merged_predictors_seasonal_data, left_on=['year', 'season'], right_on=['effect_year', 'effect_season'], how='left').drop(['effect_year', 'effect_season'], axis=1)

        # drop NAs, as we are merging one season at a time
        ml_data_seasonal.dropna(inplace=True)

        save_path = os.path.join(ml_csv_folder_save_path, f'{region}_{season}_ml_data_seasonal.csv')

        ml_data_seasonal.to_csv(save_path, index=False)

# Generate Monthly ML data
- ML data where monthly tercile category is assigned per month
- ML data where monthly tercile category is assigned by season

Example:
Monthly: If 3, 4, 5 has tercile categories BN, AN, and AN, they will be assigned BN, AN, and AN respectively.

Monthly_Seasonal_Tercile: If 3, 4, 5 has tercile categories BN, AN, and AN, but the MAM season (3,4,5) was AN (on average), then 3, 4, and 5 will all be assigned AN.

In [46]:
def create_lead_month_mapping_monthly(target_start_month, lead_time):
    """
    Calculates the prediction month for a given target season and lead time.

    Args:
        target_season_name (str): The name of the target season (e.g., 'MAM').
        target_start_month (int): The numerical start month (1-12) of the target season.
        lead_time (int): The number of months lead time (e.g., 4).

    Returns:
        dict: A dictionary mapping the prediction month (int) to the target season name (str).
              Example: {11: 'MAM'} for a 4-month lead to March.
    """
    if not 1 <= target_start_month <= 12:
        raise ValueError("target_start_month must be between 1 and 12")
    if lead_time < 0:
        raise ValueError("lead_time cannot be negative")

    # Calculate the prediction month (1-12)
    # (target_start_month - lead_time - 1) gives the zero-based index offset
    # % 12 handles the wrap-around for negative results
    # + 1 converts back to 1-based month index
    prediction_month = (target_start_month - lead_time - 1) % 12 + 1
    if prediction_month >= 12:
        prediction_month_1 = prediction_month - 11
    else:
        prediction_month_1 = prediction_month + 1
    if prediction_month >= 11:
        prediction_month_2 = prediction_month - 10
    else:
        prediction_month_2 = prediction_month + 2
    target_start_month_1 = target_start_month % 12 + 1
    target_start_month_2 = target_start_month % 12 + 2

    return {prediction_month: target_start_month, prediction_month_1: target_start_month_1, prediction_month_2: target_start_month_2}

In [47]:
# --- Define region and season info ---
region_season_start_info = {'eastern_east_africa': {'MAM': 3, 'OND': 10},
                            'lake_victoria_basin': {'DJF': 12, 'SON': 9, 'MAM': 3},
                            'west_africa': {'JAS': 7},
                            'south_sudan': {'MJJ':5, 'JAS':7, 'ASO':8},
                            'eastern_ukraine': {'JA': 7, 'AMJ': 4, 'DJF': 12},
                            'sri_lanka': {'OND': 10},
                            'southern_africa': {'FMA': 2, 'DJF': 12}}

# --- Generate all lead maps from 6 to 8 months ---
all_region_season_leads_monthly = {}

for region, seasons in region_season_start_info.items():
    all_region_season_leads_monthly[region] = {}
    for season, season_start in seasons.items():
        lead_maps = {}
        for lead in range(6, 9):  # Leads from 6 to 8 months
            lead_maps[lead] = create_lead_month_mapping_monthly(season_start, lead)
        all_region_season_leads_monthly[region][season] = lead_maps

# Result: a nested dictionary like:
# {
#     'eastern_east_africa': {
#         'MAM': {
#             6: {9: 3, 10: 4, 11: 5},
#             7: {8: 3, 9: 4, 10: 5},
#             8: {7: 3, 8: 4, 9: 5}
#         },
#         ...
#     },
#     ...
# }


In [48]:
all_region_season_leads_monthly

{'eastern_east_africa': {'MAM': {6: {9: 3, 10: 4, 11: 5},
   7: {8: 3, 9: 4, 10: 5},
   8: {7: 3, 8: 4, 9: 5}},
  'OND': {6: {4: 10, 5: 11, 6: 12},
   7: {3: 10, 4: 11, 5: 12},
   8: {2: 10, 3: 11, 4: 12}}},
 'lake_victoria_basin': {'DJF': {6: {6: 12, 7: 1, 8: 2},
   7: {5: 12, 6: 1, 7: 2},
   8: {4: 12, 5: 1, 6: 2}},
  'SON': {6: {3: 9, 4: 10, 5: 11},
   7: {2: 9, 3: 10, 4: 11},
   8: {1: 9, 2: 10, 3: 11}},
  'MAM': {6: {9: 3, 10: 4, 11: 5},
   7: {8: 3, 9: 4, 10: 5},
   8: {7: 3, 8: 4, 9: 5}}},
 'west_africa': {'JAS': {6: {1: 7, 2: 8, 3: 9},
   7: {12: 7, 1: 8, 2: 9},
   8: {11: 7, 12: 8, 1: 9}}},
 'south_sudan': {'MJJ': {6: {11: 5, 12: 6, 1: 7},
   7: {10: 5, 11: 6, 12: 7},
   8: {9: 5, 10: 6, 11: 7}},
  'JAS': {6: {1: 7, 2: 8, 3: 9},
   7: {12: 7, 1: 8, 2: 9},
   8: {11: 7, 12: 8, 1: 9}},
  'ASO': {6: {2: 8, 3: 9, 4: 10},
   7: {1: 8, 2: 9, 3: 10},
   8: {12: 8, 1: 9, 2: 10}}},
 'eastern_ukraine': {'JA': {6: {1: 7, 2: 8, 3: 9},
   7: {12: 7, 1: 8, 2: 9},
   8: {11: 7, 12: 8, 1: 9}}

In [49]:
# Assume:
# - predictors is your base DataFrame (with 'month' and 'year' columns)
# - all_region_season_lead_maps is the output from earlier automation
# - region_season_start_info is defined

processed_dfs = {}

for region, seasons in all_region_season_leads_monthly.items():
    processed_dfs[region] = {}
    for season, lead_maps in seasons.items():
        processed_dfs[region][season] = {}

        for lead_time, month_mapping in lead_maps.items():
            print(f"Processing: Region={region}, Season={season}, Lead={lead_time}")

            # 1. Copy base predictors
            temp_df = predictors.copy()

            # 2. Map effect month
            temp_df['effect_month'] = temp_df['month'].map(month_mapping)

            # 3. Compute effect year
            season_start = region_season_start_info[region][season]
            year_adjustment = 1 if lead_time >= season_start else 0

            if 'year' in temp_df.columns:
                temp_df['effect_year'] = temp_df['year'] + year_adjustment
            else:
                print(f"Warning: 'year' column missing for {region}, {season}, lead {lead_time}")

            # 4. Clean up NaNs and drop month column
            temp_df = temp_df.dropna(subset=['effect_month'])
            if 'month' in temp_df.columns:
                temp_df = temp_df.drop(columns='month')

            # 5. Store result
            processed_dfs[region][season][f'lead_time_{lead_time}'] = temp_df

Processing: Region=eastern_east_africa, Season=MAM, Lead=6
Processing: Region=eastern_east_africa, Season=MAM, Lead=7
Processing: Region=eastern_east_africa, Season=MAM, Lead=8
Processing: Region=eastern_east_africa, Season=OND, Lead=6
Processing: Region=eastern_east_africa, Season=OND, Lead=7
Processing: Region=eastern_east_africa, Season=OND, Lead=8
Processing: Region=lake_victoria_basin, Season=DJF, Lead=6
Processing: Region=lake_victoria_basin, Season=DJF, Lead=7
Processing: Region=lake_victoria_basin, Season=DJF, Lead=8
Processing: Region=lake_victoria_basin, Season=SON, Lead=6
Processing: Region=lake_victoria_basin, Season=SON, Lead=7
Processing: Region=lake_victoria_basin, Season=SON, Lead=8
Processing: Region=lake_victoria_basin, Season=MAM, Lead=6
Processing: Region=lake_victoria_basin, Season=MAM, Lead=7
Processing: Region=lake_victoria_basin, Season=MAM, Lead=8
Processing: Region=west_africa, Season=JAS, Lead=6
Processing: Region=west_africa, Season=JAS, Lead=7
Processing: R

In [50]:
# --- Assumptions ---
# processed_dfs: nested dictionary [region][season][lead] = DataFrame
# Each DataFrame has at least: ['effect_month', 'effect_year', ...]
# We will output: merged_dfs[region][season] = merged DataFrame across lead times

merge_keys = ['effect_month', 'effect_year']
merged_dfs = {}

for region, seasons in processed_dfs.items():
    merged_dfs[region] = {}

    for season, leads in seasons.items():
        renamed_dfs = []

        for lead_key, df in leads.items():
            # Extract lead number (e.g., 'lead_time_6' -> '6')
            lead = lead_key.split('_')[-1]
            suffix = f"_L{lead}"

            df_renamed = df.copy()

            if 'year' in df_renamed.columns:
                df_renamed = df_renamed.drop('year', axis=1)

            # Check merge keys
            missing_keys = [k for k in merge_keys if k not in df_renamed.columns]
            if missing_keys:
                raise ValueError(f"Missing merge keys {missing_keys} in {region}-{season} lead {lead}")

            # Rename non-merge columns
            rename_cols = {
                col: f"{col}{suffix}" for col in df_renamed.columns if col not in merge_keys
            }
            df_renamed = df_renamed.rename(columns=rename_cols)
            renamed_dfs.append(df_renamed)

        # Merge all renamed_dfs horizontally
        if not renamed_dfs:
            print(f"No data to merge for {region} {season}.")
            merged_dfs[region][season] = pd.DataFrame()
        else:
            from functools import reduce
            merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_keys, how='outer'), renamed_dfs)
            merged_dfs[region][season] = merged_df

            print(f"Merged {region} - {season} completed. Shape: {merged_df.shape}")


Merged eastern_east_africa - MAM completed. Shape: (99, 26)
Merged eastern_east_africa - OND completed. Shape: (99, 26)
Merged lake_victoria_basin - DJF completed. Shape: (99, 26)
Merged lake_victoria_basin - SON completed. Shape: (99, 26)
Merged lake_victoria_basin - MAM completed. Shape: (99, 26)
Merged west_africa - JAS completed. Shape: (102, 26)
Merged south_sudan - MJJ completed. Shape: (99, 26)
Merged south_sudan - JAS completed. Shape: (102, 26)
Merged south_sudan - ASO completed. Shape: (102, 26)
Merged eastern_ukraine - JA completed. Shape: (102, 26)
Merged eastern_ukraine - AMJ completed. Shape: (99, 26)
Merged eastern_ukraine - DJF completed. Shape: (99, 26)
Merged sri_lanka - OND completed. Shape: (99, 26)
Merged southern_africa - FMA completed. Shape: (99, 26)
Merged southern_africa - DJF completed. Shape: (99, 26)


In [51]:
# Assume:
# - merged_dfs: {region: {season: merged_monthly_df}}
# - labeled_chirps_monthly_dfs: {region: monthly_chirps_df}  # with ['year', 'month']

for region, season_dict in merged_dfs.items():
    # Get labeled CHIRPS monthly data for this region
    labeled_chirps_monthly_df = labeled_chirps_monthly_dfs[region]

    for season, merged_monthly_predictors_df in season_dict.items():

        # Merge predictors with CHIRPS monthly data on year and month
        ml_data_monthly = labeled_chirps_monthly_df.merge(
            merged_monthly_predictors_df,
            left_on=['year', 'month'],
            right_on=['effect_year', 'effect_month'],
            how='left'
        )

        # Drop unused join columns
        ml_data_monthly = ml_data_monthly.drop(['effect_year', 'effect_month'], axis=1)

        # Drop rows with missing predictor data (we merge one season at a time)
        ml_data_monthly = ml_data_monthly.dropna()

        save_path = os.path.join(ml_csv_folder_save_path, f'{region}_{season}_ml_data_monthly.csv')

        # Save to CSV
        ml_data_monthly.to_csv(
            save_path,
            index=False
        )


In [53]:
'''Generate monthly ML data where each month is assigned its respective seasonal tercile (fix)
'''

def assign_season(month, season_month_map):
    '''Helper function to assign seasons to months based on a mapping dictionary.

    Args:
        month (int): The month number (1-12).
        season_month_map (dict): A dictionary mapping season names to lists of month numbers.

    Returns:
        str: The season name if a match is found, otherwise None.
    '''
    for season_name, months in season_month_map.items():
        if month in months:
            return season_name
    return None

# for each region and chirps labeled seasonal data
for region, labeled_seasonal_data in labeled_chirps_seasonal_dfs.items():

    # assign season column to monthly data
    monthly_df = chirps_mapped_dfs[region]['monthly'].copy()
    season_month_map = region_seasons_months_dict[region]['seasons']
    monthly_df['season'] = monthly_df['month'].apply(lambda m: assign_season(m, season_month_map))

    # merge using both year and season for correct tercile alignment
    labeled_monthly = monthly_df.merge(
        labeled_seasonal_data[['year', 'season', 'tercile']],
        on=['year', 'season'],
        how='left'
    )

    for season in season_month_map.keys():
        # get the monthly predictors for this season
        current_region_monthly_predictors = merged_dfs[region][season]

        # merge predictors
        ml_data = labeled_monthly.merge(
            current_region_monthly_predictors,
            left_on=['year', 'month'],
            right_on=['effect_year', 'effect_month'],
            how='left'
        ).drop(['effect_year', 'effect_month'], axis=1)

        ml_data.dropna(inplace=True)
        ml_data.drop_duplicates(inplace=True)

        # save
        save_path = os.path.join(ml_csv_folder_save_path, f'{region}_{season}_ml_data_monthly_seasonal_tercile.csv')
        ml_data.to_csv(save_path, index=False)
